In [ ]:
import os
import pandas as pd

## first set of fractions
fractions = [1.0, 0.1, 0.01, 0.001]
# map the fractions to sheet names of the data stored
sheet_map = {
    1.0: "100%",
    0.1: "10%",
    0.01: "1%",
    0.001: "0%", 
}

def load_all_pfba_models(folder):
    all_models = {}

    for fname in os.listdir(folder):
        if not fname.endswith(".xlsx") or fname.startswith("~$"):
            continue

        model_id = fname.replace("_pfba.xlsx", "")
        fpath = os.path.join(folder, fname)

        try:
            xls = pd.ExcelFile(fpath, engine="openpyxl")
        except Exception as e:
            print(f"⚠️ Skipping file {fname}: {e}")
            continue

        model_data = {}
        for f in fractions:
            sheet = sheet_map[f]
            if sheet in xls.sheet_names:
                df = pd.read_excel(fpath, sheet_name=sheet, engine="openpyxl", header=0)

                # standardize column names (in case of whitespace)
                df.columns = [str(c).strip() for c in df.columns]

                model_data[f] = df
            else:
                model_data[f] = pd.DataFrame(columns=["Reactions","fluxes","names"])

        all_models[model_id] = model_data

    return all_models


In [ ]:
import os
import pandas as pd

## second set of fractions
fractions_2_5 = [0.05, 0.02]
sheet_map_2_5 = {0.05: "5%", 0.02: "2%"}

def load_pfba_2_5(folder):
    all_models = {}

    for fname in os.listdir(folder):
        if not fname.endswith(".xlsx") or fname.startswith("~$"):
            continue

        model_id = fname.replace("_pfba.xlsx", "")
        fpath = os.path.join(folder, fname)

        try:
            xls = pd.ExcelFile(fpath, engine="openpyxl")
        except Exception as e:
            print(f"⚠️ Skipping {fname}: {e}")
            continue

        model_data = {}
        for frac in fractions_2_5:
            sh = sheet_map_2_5[frac]
            if sh in xls.sheet_names:
                df = pd.read_excel(xls, sh)
                # normalize colnames
                df.columns = [str(c).strip() for c in df.columns]
                model_data[frac] = df
            else:
                model_data[frac] = pd.DataFrame(columns=["Reactions","fluxes","names"])

        all_models[model_id] = model_data

    return all_models

In [ ]:
import os
import pandas as pd

## third set of fractions
fractions_20_50 = [0.5, 0.2]
sheet_map_20_50 = {0.5: "50%", 0.2: "20%"}

def load_pfba_20_50(folder):
    all_models = {}

    for fname in os.listdir(folder):
        if not fname.endswith(".xlsx") or fname.startswith("~$"):
            continue

        model_id = fname.replace("_pfba.xlsx", "")
        fpath = os.path.join(folder, fname)

        try:
            xls = pd.ExcelFile(fpath, engine="openpyxl")
        except Exception as e:
            print(f"⚠️ Skipping {fname}: {e}")
            continue

        model_data = {}
        for frac in fractions_20_50:
            sh = sheet_map_20_50[frac]
            if sh in xls.sheet_names:
                df = pd.read_excel(xls, sh)
                # normalize colnames
                df.columns = [str(c).strip() for c in df.columns]
                model_data[frac] = df
            else:
                model_data[frac] = pd.DataFrame(columns=["Reactions","fluxes","names"])

        all_models[model_id] = model_data

    return all_models

In [ ]:
import os
import pandas as pd

## third set of fractions
fractions_02_05 = [0.005, 0.002]
sheet_map_02_05 = {0.005: "0.005", 0.002: "0.002"}

def load_pfba_02_05(folder):
    all_models = {}

    for fname in os.listdir(folder):
        if not fname.endswith(".xlsx") or fname.startswith("~$"):
            continue

        model_id = fname.replace("_pfba.xlsx", "")
        fpath = os.path.join(folder, fname)

        try:
            xls = pd.ExcelFile(fpath, engine="openpyxl")
        except Exception as e:
            print(f"⚠️ Skipping {fname}: {e}")
            continue

        model_data = {}
        for frac in fractions_02_05:
            sh = sheet_map_02_05[frac]
            if sh in xls.sheet_names:
                df = pd.read_excel(xls, sh)
                # normalize colnames
                df.columns = [str(c).strip() for c in df.columns]
                model_data[frac] = df
            else:
                model_data[frac] = pd.DataFrame(columns=["Reactions","fluxes","names"])

        all_models[model_id] = model_data

    return all_models

In [ ]:
# store data from each excel into the folowing 4 dataframes
pfba_data_1 = load_all_pfba_models("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_pfba/")
pfba_data_2 = load_pfba_2_5("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_pfba1/")
pfba_data_3 = load_pfba_20_50("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_pfba2/")
pfba_data_4 = load_pfba_02_05("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Pediatric_pfba3/")

In [ ]:
import numpy as np
# add group name to the reactions in each dataframe for feature generation
def ensure_groups_column(pfba_data, fill_value=np.nan):
    for model, frac_dict in pfba_data.items():
        for frac, df in frac_dict.items():
            if df is None or df.empty:
                continue

            if "groups" not in df.columns:
                df["groups"] = fill_value

    return pfba_data


In [ ]:
pfba_data_1 = ensure_groups_column(pfba_data_1)
pfba_data_2 = ensure_groups_column(pfba_data_2)
pfba_data_3 = ensure_groups_column(pfba_data_3)
pfba_data_4 = ensure_groups_column(pfba_data_4)

# check the dataframes
m = "ACH-000312"
print(pfba_data_2[m][0.02].shape)
print(pfba_data_3[m][0.2].columns)
print(pfba_data_1[m][0.001].head())


In [ ]:
# add group names to reactions (from Recon 3D)
def map_groups_from_excel(
    pfba_data,
    excel_df,
    rxn_col_excel="Reactions",
    group_col_excel="groups"
):
    # Build fast lookup dict: reaction_id -> group
    rxn_to_group = (
        excel_df[[rxn_col_excel, group_col_excel]]
        .dropna(subset=[rxn_col_excel])
        .set_index(rxn_col_excel)[group_col_excel]
        .to_dict()
    )

    for model, frac_dict in pfba_data.items():
        for frac, df in frac_dict.items():
            if df is None or df.empty:
                continue

            # overwrite / create groups cleanly
            df["groups"] = df["Reactions"].map(rxn_to_group)

    return pfba_data


In [ ]:
# Load your Recon 3D excel for mapping
excel_map = pd.read_excel("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Recon3D_groups.xlsx")

# Apply mapping
pfba_data_1 = map_groups_from_excel(
    pfba_data_1,
    excel_df=excel_map,
    rxn_col_excel="Reactions",
    group_col_excel="groups"
)



In [ ]:
# Load your Recon 3D excel for mapping
excel_map = pd.read_excel("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Recon3D_groups.xlsx")

# Apply mapping
pfba_data_2 = map_groups_from_excel(
    pfba_data_2,
    excel_df=excel_map,
    rxn_col_excel="Reactions",
    group_col_excel="groups"
)


In [ ]:
# Load your Recon 3D excel for mapping
excel_map = pd.read_excel("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Recon3D_groups.xlsx")

# Apply mapping
pfba_data_3 = map_groups_from_excel(
    pfba_data_3,
    excel_df=excel_map,
    rxn_col_excel="Reactions",
    group_col_excel="groups"
)


In [ ]:
# Load your Recon 3D excel for mapping
excel_map = pd.read_excel("/Users/subasrees/Downloads/Frontiers/Frontier_ccle/Recon3D_groups.xlsx")

# Apply mapping
pfba_data_4 = map_groups_from_excel(
    pfba_data_4,
    excel_df=excel_map,
    rxn_col_excel="Reactions",
    group_col_excel="groups"
)


In [ ]:
## check whether group names are added
m = list(pfba_data_3.keys())[0]
f = sorted(pfba_data_3[m].keys())[0]
df = pfba_data_3[m][f]
df


In [ ]:
# function to add group names of RS reactions merged from RS model
def add_ros_rss_groups_to_pfba(pfba_data):
    for model_id, model_data in pfba_data.items():
        for frac, df in model_data.items():
            if df is None or df.empty:
                continue

            # Only proceed if expected columns exist
            if "Reactions" not in df.columns or "groups" not in df.columns:
                continue

            df = df.copy()
            df.loc[df["Reactions"].str.endswith("demand", na=False), "groups"] = "RS_demand" 
            df.loc[df["Reactions"] == "H2O2_m_demand", "groups"] = "Mito_ROS_demand"
            df.loc[df["Reactions"] == "H2O2_x_demand", "groups"] = "Perox_ROS_demand"
            df.loc[df["Reactions"] == "oh_rad_c_demand", "groups"] = "Cyto_ROS_demand"
            df.loc[df["Reactions"] == "H2S_m_demand", "groups"] = "Mito_RSS_demand"
            df.loc[df["Reactions"] == "H2S_c_demand", "groups"] = "Cyto_RSS_demand"
            df.loc[df["Reactions"] == "HS_c_demand", "groups"] = "Cyto_RSS_demand"
            df.loc[df["Reactions"] == "Hypochlorous_c_demand", "groups"] = "Cyto_RHS_demand"
            df.loc[df["groups"] == "mem_transports", "groups"]="Transport, extracellular"
            df.loc[df["Reactions"] == "DM_h2o2[total]", "groups"] = "ROS_demand_total"
            df.loc[df["Reactions"] == "DM_oh_rad[total]", "groups"] = "ROS_demand_total"
            df.loc[df["Reactions"] == "DM_h2s[total]", "groups"] = "RSS_demand_total"
            df.loc[df["Reactions"] == "DM_HC00250[total]", "groups"] = "RSS_demand_total"
            df.loc[df["Reactions"] == "DM_CE4633[total]", "groups"] = "RHS_demand_total"
            df.loc[df["Reactions"].str.startswith("EX_", na=False), "groups"] = "Exchange"
            df.loc[df["Reactions"].str.startswith("sink_", na=False), "groups"] = "Sinks"
            # everything else that starts with DM_ becomes generic Demand
            mask = df["Reactions"].str.startswith("DM_", na=False) & ~df["Reactions"].isin(
                ["DM_h2o2[total]", "DM_oh_rad[total]", "DM_h2s[total]","DM_CE4633[total]","DM_HC00250[total]"]
            )
            df.loc[mask, "groups"] = "Demand"
            model_data[frac] = df  # write back
    return pfba_data


In [ ]:
pfba_data_1 = add_ros_rss_groups_to_pfba(pfba_data_1)
pfba_data_2 = add_ros_rss_groups_to_pfba(pfba_data_2)
pfba_data_3 = add_ros_rss_groups_to_pfba(pfba_data_3)
pfba_data_4 = add_ros_rss_groups_to_pfba(pfba_data_4)

In [ ]:
import pandas as pd
# function to map group names of RS reactions merged from RS model, from RS model excel 

def add_rs_groups_from_excel(pfba_data, excel_path):
    rs_map = pd.read_excel(excel_path)

    # make a dict for fast lookup
    rs_dict = dict(zip(rs_map["Abbreviation"], rs_map["Subsystem"]))

    for model_id, model_data in pfba_data.items():
        for frac, df in model_data.items():
            if df is None or df.empty:
                continue
            if "Reactions" not in df.columns or "groups" not in df.columns:
                continue

            df = df.copy()
            mask = df["Reactions"].astype(str).str.startswith("RS_", na=False)

            # map group names where available
            df.loc[mask, "groups"] = df.loc[mask, "Reactions"].map(rs_dict).fillna(df.loc[mask, "groups"])

            model_data[frac] = df

    return pfba_data


In [ ]:
pfba_data_1 = add_rs_groups_from_excel(pfba_data_1, "/Users/subasrees/Downloads/Frontiers/Frontier_ccle/RSmodel_Recon3D1_2024_withGPRs_updated.xls")
pfba_data_2 = add_rs_groups_from_excel(pfba_data_2, "/Users/subasrees/Downloads/Frontiers/Frontier_ccle/RSmodel_Recon3D1_2024_withGPRs_updated.xls")
pfba_data_3 = add_rs_groups_from_excel(pfba_data_3, "/Users/subasrees/Downloads/Frontiers/Frontier_ccle/RSmodel_Recon3D1_2024_withGPRs_updated.xls")
pfba_data_4 = add_rs_groups_from_excel(pfba_data_4, "/Users/subasrees/Downloads/Frontiers/Frontier_ccle/RSmodel_Recon3D1_2024_withGPRs_updated.xls")
# check the dataframes
m = list(pfba_data_1.keys())[0]
f = sorted(pfba_data_3[m].keys())[0]
df = pfba_data_3[m][f]

df[df["Reactions"].str.startswith("RS_", na=False)]["groups"].value_counts().head(20)



In [ ]:
import pandas as pd
import numpy as np
## function to calculate flux allocation of pathways (pathway flux/total flux)
def pathway_flux_share_matrix(pfba_data, tol=1e-7, use_abs=True):

    rows = []

    for model_id, model_data in pfba_data.items():
        for frac, df in model_data.items():

            if df is None or df.empty:
                continue

            df = df.copy()
            df.columns = [str(c).strip() for c in df.columns]

            if not {"fluxes", "groups"}.issubset(df.columns):
                continue

            df["fluxes"] = pd.to_numeric(df["fluxes"], errors="coerce").fillna(0.0)
            df = df[df["fluxes"].abs() > tol]
            df = df[df["groups"].notna()]

            if df.empty:
                continue

            v = df["fluxes"].abs() if use_abs else df["fluxes"]
            total = float(v.sum())

            if total <= 0:
                continue

            pathway_sum = v.groupby(df["groups"]).sum()
            flux_share = pathway_sum / total

            out = flux_share.to_dict()
            out["Model"] = model_id
            out["Fraction"] = float(frac)

            rows.append(out)

    long = pd.DataFrame(rows).fillna(0.0)

    X = (
        long
        .set_index(["Model", "Fraction"])
        .sort_index()
    )

    return X, long


In [ ]:
X1, long1 = pathway_flux_share_matrix(pfba_data_1)      # 4 fractions
X2, long2 = pathway_flux_share_matrix(pfba_data_2)    # 2 fractions
X3, long3 = pathway_flux_share_matrix(pfba_data_3)    # 2 fractions
X4, long4 = pathway_flux_share_matrix(pfba_data_4)    # 2 fractions

In [ ]:
# 1) Get the union of ALL columns across the three dataframes
all_cols = sorted(
    set(X1.columns)
    .union(set(X2.columns))
    .union(set(X3.columns))
    .union(set(X4.columns))
)

# 2) Reindex each dataframe to the same column set
X1u = X1.reindex(columns=all_cols, fill_value=0.0)
X2u = X2.reindex(columns=all_cols, fill_value=0.0)
X3u = X3.reindex(columns=all_cols, fill_value=0.0)
X4u = X4.reindex(columns=all_cols, fill_value=0.0)

# 3) Concatenate and align indices
X_all = pd.concat([X1u, X2u, X3u, X4u]).sort_index()

# 4) Flatten index if needed
X_all_flat = X_all.reset_index()

print("Shape:", X_all_flat.shape)
print("Any NaNs:", X_all_flat.isna().sum().sum())


In [ ]:
## define a panel of 10 oncometabolites
oncomets = [
    'EX_akg[e]', 'EX_lac_L[e]', 'EX_succ[e]', 'EX_fum[e]',
    'EX_gln_L[e]', 'EX_glu_L[e]', 'EX_ser_L[e]', 'EX_gly[e]',
    'EX_amet[e]', 'EX_ahcys[e]'
]

In [ ]:
import numpy as np
import pandas as pd
# function to calculate the ratio of fluxes of oncometabolites to the total exchange flux
def exchange_onco_features(
    pfba_data,
    oncomets,
    use_abs=True,
    tol=1e-7,
    eps=1e-12,
    debug=False
):
    rows = []

    for model, frac_dict in pfba_data.items():
        for frac, df in frac_dict.items():
            if df is None or df.empty:
                continue

            d = df.copy()

            # --- flux vector ---
            flux = pd.to_numeric(d["fluxes"], errors="coerce")
            if use_abs:
                flux = flux.abs()
            flux = flux.fillna(0.0)

            # --- masks ---
            rxn = d["Reactions"].astype(str)
            is_ex = rxn.str.startswith("EX_", na=False)
            is_onco = rxn.isin(oncomets)

            # --- totals ---
            total_flux = float(flux.sum())
            ex_flux = float(flux[is_ex].sum())
            onco_flux = float(flux[is_ex & is_onco].sum())
            other_flux = ex_flux - onco_flux

            # numerical guard
            if other_flux < 0 and abs(other_flux) < 1e-9:
                other_flux = 0.0

            # if total_flux is ~0, just output zeros
            if total_flux <= tol:
                out = {
                    "Model": model,
                    "Fraction": float(frac),

                    "Total_abs": 0.0,
                    "Exchange_abs": 0.0,
                    "OncoEX_abs": 0.0,
                    "Exchange_other_abs": 0.0,

                    # normalized by total network flux
                    "Exchange_norm_total": 0.0,
                    "OncoEX_norm_total": 0.0,
                    "Exchange_other_norm_total": 0.0,

                    # normalized by exchange flux
                    "OncoEX_norm_exchange": 0.0,
                    "Other_norm_exchange": 0.0,
                }
                rows.append(out)
                continue

            # normalized by total network flux (your original intent)
            exchange_norm_total = ex_flux / (total_flux + eps)
            onco_norm_total = onco_flux / (total_flux + eps)
            other_norm_total = other_flux / (total_flux + eps)

            # normalized by exchange only (matches your manual ratio idea)
            # This is well-defined even if ex_flux==0
            onco_norm_exchange = onco_flux / (ex_flux + eps)
            other_norm_exchange = other_flux / (ex_flux + eps)

            if debug:
                print("Model", model, "Frac", frac)
                print("total_flux", total_flux)
                print("ex_flux   ", ex_flux)
                print("onco_flux ", onco_flux)
                print("onco/ex   ", onco_norm_exchange)
                print("onco/total", onco_norm_total)
                print("total/ex  ", total_flux / (ex_flux + eps))
                print("----")

            out = {
                "Model": model,
                "Fraction": float(frac),

                "Total_abs": total_flux,
                "Exchange_abs": ex_flux,
                "OncoEX_abs": onco_flux,
                "Exchange_other_abs": other_flux,

                "Exchange_norm_total": exchange_norm_total,
                "OncoEX_norm_total": onco_norm_total,
                "Exchange_other_norm_total": other_norm_total,

                "OncoEX_norm_exchange": onco_norm_exchange,
                "Other_norm_exchange": other_norm_exchange,
            }
            rows.append(out)

    return pd.DataFrame(rows)


In [ ]:
ex_feat_1 = exchange_onco_features(pfba_data_1,oncomets)
ex_feat_2 = exchange_onco_features(pfba_data_2,oncomets)
ex_feat_3 = exchange_onco_features(pfba_data_3,oncomets)
ex_feat_4 = exchange_onco_features(pfba_data_4,oncomets)

ex_feat_all = pd.concat([ex_feat_1, ex_feat_2, ex_feat_3, ex_feat_4], ignore_index=True)
print("Shape:", ex_feat_all.shape)
print("Any NaNs:", ex_feat_all.isna().sum().sum())
X_all_with_onco = X_all_flat.merge(
    ex_feat_all[["Model","Fraction",
                 "OncoEX_norm_exchange"]],
    on=["Model","Fraction"],
    how="left"
)
# Retain only the ratio of oncometabolite flux to total flux
X_all_with_onco[["OncoEX_norm_exchange"]] = X_all_with_onco[["OncoEX_norm_exchange"]].fillna(0.0)


In [ ]:
import numpy as np
import pandas as pd
# function to add log ratios of two reaction groups, in each group there are reactions that have flux in backward direction because of stoichiometry in Recon 3DC model
def reaction_set_or_sum(df, rxn_spec, flux_col="fluxes", rxn_col="Reactions",
                        tol=1e-7, use_abs=True, pick="max_abs"):
    df = df.copy()
    df[flux_col] = pd.to_numeric(df[flux_col], errors="coerce").fillna(0.0)

    # apply tolerance
    df = df[df[flux_col].abs() > tol].copy()

    # map reaction -> flux
    flux_series = df.set_index(rxn_col)[flux_col]
    if use_abs:
        flux_series = flux_series.abs()

    flux_sum = 0.0
    count_present = 0
    slots = len(rxn_spec)

    for item in rxn_spec:
        # single reaction
        if isinstance(item, str):
            if item in flux_series.index:
                flux_sum += float(flux_series.loc[item])
                count_present += 1
            continue

        # OR group
        if isinstance(item, (list, tuple, set)):
            candidates = [r for r in item if r in flux_series.index]
            if not candidates:
                continue

            if pick == "first":
                chosen = candidates[0]
            else:  # "max_abs"
                chosen = max(candidates, key=lambda r: float(flux_series.loc[r]))

            flux_sum += float(flux_series.loc[chosen])
            count_present += 1
            continue

        raise TypeError(f"Unsupported element type in rxn_spec: {type(item)}")

    coverage = count_present / slots if slots > 0 else 0.0
    return flux_sum, count_present, coverage


In [ ]:
import numpy as np
import pandas as pd
# ATP production from glycolysis vs. total central carbon metabolism
def sum_producing_flux(df, rxns, prod_sign_map, col_rxn="Reactions", col_flux="fluxes"):
    """
    Sum only the ATP-producing component of flux for a set of reactions.
    prod_sign_map[rxn] = +1 means positive flux produces ATP
    prod_sign_map[rxn] = -1 means negative flux produces ATP
    """
    if df is None or df.empty:
        return 0.0

    sub = df[df[col_rxn].isin(rxns)]
    if sub.empty:
        return 0.0

    total = 0.0
    for r, f in zip(sub[col_rxn].astype(str).values, sub[col_flux].astype(float).values):
        s = prod_sign_map.get(r, +1)   # default: forward produces ATP
        total += max(0.0, s * f)       # keep only producing-direction component
    return float(total)


In [ ]:
def build_ratio_features_pfba(
    pfba_data,
    ratio_module,
    prod_sign_map,
    eps=1e-7,
    log_base=2
):
    rows = []

    for model, frac_dict in pfba_data.items():
        for frac, df in frac_dict.items():
            if df is None or df.empty:
                continue

            out = {"Model": model, "Fraction": float(frac)}

            for feat_name, (num_rxns, den_rxns) in ratio_module.items():
                num = sum_producing_flux(df, num_rxns, prod_sign_map)
                den = sum_producing_flux(df, den_rxns, prod_sign_map)

                # log ratio
                if log_base == 2:
                    out[f"{feat_name}__log_ratio"] = np.log2((num + eps) / (den + eps))
                else:
                    out[f"{feat_name}__log_ratio"] = np.log((num + eps) / (den + eps))

            rows.append(out)

    return pd.DataFrame(rows)


In [ ]:
prod_sign_map = {
    "PGK": -1,      # negative flux produces ATP 
    "SUCOASm": -1,  # negative flux produces ATP

    "PYK": +1,      # forward produces ATP
    "ATPS4mi": +1,  # forward produces ATP
}


In [ ]:
ratio_module_atp = {
    "ATP_Gly_vs_ATP_Carbon": (
        ["PGK", "PYK"],                       
        ["PGK", "PYK", "ATPS4mi", "SUCOASm"]   
    )
}
ratio_feat_df_1 = build_ratio_features_pfba(
    pfba_data=pfba_data_1,
    ratio_module=ratio_module_atp,
    prod_sign_map=prod_sign_map,
    eps=1e-7
)
ratio_feat_df_2 = build_ratio_features_pfba(
    pfba_data=pfba_data_2,
    ratio_module=ratio_module_atp,
    prod_sign_map=prod_sign_map,
    eps=1e-7
)
ratio_feat_df_3 = build_ratio_features_pfba(
    pfba_data=pfba_data_3,
    ratio_module=ratio_module_atp,
    prod_sign_map=prod_sign_map,
    eps=1e-7
)
ratio_feat_df_4 = build_ratio_features_pfba(
    pfba_data=pfba_data_4,
    ratio_module=ratio_module_atp,
    prod_sign_map=prod_sign_map,
    eps=1e-7
)

In [ ]:
atp_feat = pd.concat([ratio_feat_df_1, ratio_feat_df_2, ratio_feat_df_3,ratio_feat_df_4], ignore_index=True)


In [ ]:
## define pathway ratio modules that need to be included as features
ratio_modules = {
    # module_name: (numerator_spec, denominator_spec)
    #"ATP_Gly_vs_ATP_Carbon": (
    #    ["PGK", "PYK"],                
    #    ["PGK", "PYK","ATPS4mi"]             
    #),
    "Glutaminolysis_vs_Total_TCA_input": (
        [["GLUDxm","GLUDym"], "GLUNm", "ASPTA", "ALATA_L"],  # OR group included
        ["PCm", "PDHm",["GLUDxm","GLUDym"], "GLUNm", "ASPTA", "ALATA_L"]
    ),
    "PPP_vs_Glycolysis":([["G6PDH1rer","G6PDH2rer"],"PGLer","GNDer"],["HEX1","PGI", "PFK", "FBA", "GAPD", "PGK", "PGM", "ENO", "PYK"]),
    #"FA_synthesis_vs_β‑oxidation":(['ACCOACm',	'FAS100COA',	'FAS120COA',	'FAS140COA',	'FAS160COA',	'FAS180',	'FAS180COA',	'FAS80COA_L',	'DESAT14_9',	'DESAT16_2',	'DESAT18_10',	'DESAT18_3',	'DESAT18_4',	'DESAT18_5',	'DESAT18_6',	'DESAT18_7',	'DESAT18_8',	'DESAT18_9',	'DESAT20_1',	'DESAT20_2',	'DESAT22_1p',	'DESAT22_2p',	'DESAT24_1'],['TMNDNCCPT1',	'TETTET6CPT1',	'TETPENT6CPT1',	'TETPENT3CPT1',	'STRDNCCPT1',	'NRVNCCPT1',	'LNLNCGCPT1',	'LNLNCACPT1',	'LNLCCPT1',	'LNELDCCPT1',	'LGNCCPT1',	'HMR_2877',	'HMR_2774',	'HMR_2771',	'HMR_2768',	'HMR_2739',	'HMR_2736',	'HMR_2733',	'HMR_2711',	'HMR_2708',	'HMR_2705',	'HMR_2702',	'HMR_2699',	'HMR_2693',	'HMR_2690',	'HMR_2687',	'HMR_2684',	'HMR_2681',	'HMR_2675',	'HMR_2669',	'HMR_2666',	'HMR_2660',	'HMR_2657',	'HMR_2651',	'HMR_2648',	'HMR_2633',	'HMR_2620',	'HMR_2614',	'HMR_2611',	'HMR_2608',	'HMR_2605',	'HMR_2602',	'HEXCCPT1',	'ELAIDCPT1',	'EICOSTETCPT1',	'DLNLCGCPT1',	'DCSPTN1CPT1',	'CLPNDCPT1',	'C161CPT12',	'ADRNCPT1',	'HMR_2930',	'HMR_2918',	'HMR_2906',	'C180CPT2',	'r1481',	'r1477',	'r1474',	'r1472',	'RE3073X',	'r1487',	'RE3160R',	'RE3146R',	'RE3132R',	'RE3114R',	'RE3074X',	'RE2912M',	'r1262',	'r1260',	'r1259',	'r1257',	'r1255',	'r1254',	'r1253',	'r1252',	'r1251',	'HMR_3522',	'HMR_3244',	'HMR_3240',	'HMR_3234',	'HMR_3230',	'HMR_3222',	'HMR_3218',	'HMR_3198',	'HMR_3194',	'HMR_3190',	'HMR_3186',	'HMR_3182',	'HMR_3178',	'HMR_3174',	'HMR_3170',	'HMR_3149',	'HMR_3142',	'HMR_3135',	'HMR_3128',	'HMR_3121',	'HMR_3115',	'HMR_3111',	'HMR_3107',	'HMR_3454',	'HMR_3450']),
    #"Cholesterol_synthesis_vs_FA_synthesis":(['MEVK1c',	'PMEVKc',	'DPMVDc',	'IPDDI',	'DMATT',	'GRTT',	'FRDPtcr',	'SQLSr',	'SQLEr',	'LNSTLSr',	'r0781',	'C14STRr',	'C4STMO1r',	'C3STDH1r',	'C3STDH1Pr',	'C4STMO2r',	'C4STMO2Pr',	'C3STKR2r',	'DHCR241r',	'EBP2r',	'LSTO2r',	'DHCR72r'],['ACCOACm',	'FAS100COA',	'FAS120COA',	'FAS140COA',	'FAS160COA',	'FAS180',	'FAS180COA',	'FAS80COA_L',	'DESAT14_9',	'DESAT16_2',	'DESAT18_10',	'DESAT18_3',	'DESAT18_4',	'DESAT18_5',	'DESAT18_6',	'DESAT18_7',	'DESAT18_8',	'DESAT18_9',	'DESAT20_1',	'DESAT20_2',	'DESAT22_1p',	'DESAT22_2p',	'DESAT24_1']),
    #"Purine_denovo_vs_Total_purine":(['PRPPS',	'GLUPRT',	'GARFT',	'PRAGSr',	'PRFGS',	'r0666',	'AIRCr',	'PRASCS'],['PRPPS',	'GLUPRT',	'GARFT',	'PRAGSr',	'PRFGS',	'r0666',	'AIRCr',	'PRASCS', 'ADPT',	'GUAPRT',	'HXPRT']),
    #"Pyrimidine_denovo_vs_Total_Pyrimidine":(['ASPCTr',	'DHORD9',	'ORPT',	'CBPS',	'CTPS2',	'DHORTS',	'OMPDC',	'UMPK',	'NDPK2',	'CTPS1'],['ADSL2',	'AICART',	'IMPC',	'IMPD',	'GMPS2',	'NDPK1',	'ADSS',	'ADSL1','ASPCTr',	'DHORD9',	'ORPT',	'CBPS',	'CTPS2',	'DHORTS',	'OMPDC',	'UMPK',	'NDPK2',	'CTPS1']),
    "Serine_fraction":([['GHMT2rm','GHMT2r']],['FOLR2',	['MTHFD',	'MTHFD2'],	['FTHFL',	'FTHFLm']]),
    #"Purine_folate_fraction":(['GARFT','AICART','IMPC'],['FOLR2',	['MTHFD',	'MTHFD2'],	['FTHFL',	'FTHFLm']]),
    # "dTMP_folate_fraction":(['TMDS'],['FOLR2',	['MTHFD',	'MTHFD2'],	['FTHFL',	'FTHFLm']]),
    #"RSS_ROS_fraction":([],[])
}


In [ ]:
def build_features_hybrid(pfba_data, ratio_modules, tol=1e-7, use_abs=True, pick="max_abs", eps=1e-12):
    all_rows = []

    for model_id, model_data in pfba_data.items():
        for frac, df in model_data.items():
            row = {"Model": model_id, "Fraction": frac}

            if df is None or df.empty:
                for mod in ratio_modules.keys():
                    row[f"R_{mod}__num_flux"] = 0.0
                    row[f"R_{mod}__den_flux"] = 0.0
                    row[f"R_{mod}__value"] = 0.0
                    row[f"R_{mod}__log_ratio"] = 0.0
                    row[f"R_{mod}__num_zero"] = 1
                    row[f"R_{mod}__den_zero"] = 1
                    row[f"R_{mod}__both_zero"] = 1
                    row[f"R_{mod}__num_count"] = 0
                    row[f"R_{mod}__den_count"] = 0
                    row[f"R_{mod}__num_cov"] = 0.0
                    row[f"R_{mod}__den_cov"] = 0.0
                all_rows.append(row)
                continue

            for mod, (num_spec, den_spec) in ratio_modules.items():
                num_flux, num_count, num_cov = reaction_set_or_sum(df, num_spec, tol=tol, use_abs=use_abs, pick=pick)
                den_flux, den_count, den_cov = reaction_set_or_sum(df, den_spec, tol=tol, use_abs=use_abs, pick=pick)

                num_zero = int(num_flux <= tol)
                den_zero = int(den_flux <= tol)
                both_zero = int(num_zero and den_zero)

                ratio_val = (num_flux / den_flux) if (den_flux > tol) else 0.0
                log_ratio = np.log10((num_flux + eps) / (den_flux + eps))

                #row[f"R_{mod}__num_flux"] = num_flux
                #row[f"R_{mod}__den_flux"] = den_flux
                #row[f"R_{mod}__value"] = ratio_val
                row[f"R_{mod}__log_ratio"] = log_ratio
                #row[f"R_{mod}__num_zero"] = num_zero
                #row[f"R_{mod}__den_zero"] = den_zero
                #row[f"R_{mod}__both_zero"] = both_zero
                #row[f"R_{mod}__num_count"] = num_count
                #row[f"R_{mod}__den_count"] = den_count
                #row[f"R_{mod}__num_cov"] = num_cov
                #row[f"R_{mod}__den_cov"] = den_cov

            all_rows.append(row)

    return pd.DataFrame(all_rows).fillna(0.0)


In [ ]:
df_hyb_1   = build_features_hybrid(pfba_data_1, ratio_modules, tol=1e-7, use_abs=True, eps=1e-12)
df_hyb_2   = build_features_hybrid(pfba_data_2, ratio_modules, tol=1e-7, use_abs=True, eps=1e-12)
df_hyb_3   = build_features_hybrid(pfba_data_3, ratio_modules, tol=1e-7, use_abs=True, eps=1e-12)
df_hyb_4   = build_features_hybrid(pfba_data_4, ratio_modules, tol=1e-7, use_abs=True, eps=1e-12)


imp_feat_all = pd.concat([df_hyb_1, df_hyb_2, df_hyb_3,df_hyb_4], ignore_index=True)


In [ ]:
feat_df=X_all_with_onco.merge(imp_feat_all,on=["Model","Fraction"],how="left")
feat_all=feat_df.merge(atp_feat,on=["Model","Fraction"],how="left")
## remove the following features - to prevent redundancy and "Oxidative phosphorylation" to prevent leakage for the ML model
feat_all=feat_all.drop(columns=["Oxidative phosphorylation","Exchange","Demand","Exchange/demand reaction"])


In [ ]:
import pandas as pd

# mapping cancer primary categories to the dataset for information, based on mapping between sample name and cancer name from an excel sheet downloaded from the cancer dependancy map
mapping_df = pd.read_excel('/Users/subasrees/Downloads/Frontiers/5_cancers/celllines.xlsx','Final')
main_df = pd.DataFrame({
    'ModelID': mapping_df['ModelID']
})
model_labels = dict(zip(mapping_df['ModelID'], mapping_df['OncotreePrimaryDisease']))#column names in excel
#print(model_labels)
main_df['OncotreePrimaryDisease'] = main_df['ModelID'].map(model_labels)
main_df=main_df.loc[:,['ModelID',  'OncotreePrimaryDisease']]
main_df.columns=['Model','Cancer']
feat_all['Cancer'] = feat_all['Model'].map(model_labels)
feature_matrix=feat_all
feature_matrix.to_excel('/Users/subasrees/Pediatric-cancer-analysis/feature_matrix_RS.xlsx')